In [ ]:
!pip install mne
!pip install tensorflow
import mne
from mne.decoding import CSP
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense


In [ ]:
with open("chb01-summary.txt", "r") as f:
    summary_text = f.read()

def get_seizure_times(summary_text, target_file):
    lines = summary_text.splitlines()
    seizure_times = []
    in_target = False

    for i, line in enumerate(lines):
        line = line.strip()

        if line.startswith("File Name:"):
            current_file = line.split(":")[1].strip()
            in_target = (current_file == target_file)

        if in_target:
            if "Number of Seizures in File: 0" in line:
                return []
            if "Seizure Start Time:" in line:
                start = int(line.split(":")[1].split()[0])
                end = int(lines[i+1].split(":")[1].split()[0])
                seizure_times.append((start, end))

    return seizure_times

In [ ]:
def process_file(file_path, seizure_times):
    raw = mne.io.read_raw_edf(file_path, preload=True)

    raw.resample(128)

    epochs = mne.make_fixed_length_epochs(raw, duration=2)
    data = epochs.get_data()

    labels = []

    PREICTAL_WINDOW = 300

    for t in epochs.events[:, 0] / raw.info['sfreq']:
        label = 0
        for start, end in seizure_times:
            preictal_start = max(0, start - PREICTAL_WINDOW)

            if preictal_start <= t < start:
                label = 1

        labels.append(label)

    return data, np.array(labels)

In [ ]:
def build_dataset(file_list):
    X_all = []
    y_all = []

    for file in file_list:
        seizure_times = get_seizure_times(summary_text, file)
        X, y = process_file(file, seizure_times)

        X_all.append(X)
        y_all.append(y)

    return np.concatenate(X_all), np.concatenate(y_all)

In [ ]:
train_files = [
    "chb01_01.edf", "chb01_02.edf", "chb01_03.edf",
    "chb01_05.edf", "chb01_06.edf", "chb01_07.edf",
    "chb01_08.edf", "chb01_09.edf"
]

test_files = [
    "chb01_04.edf", "chb01_10.edf", "chb01_11.edf", "chb01_12.edf"
]

In [ ]:
X_train_raw, y_train = build_dataset(train_files)
X_test_raw, y_test = build_dataset(test_files)

print("Train labels:", np.unique(y_train, return_counts=True))
print("Test labels:", np.unique(y_test, return_counts=True))

Extracting EDF parameters from chb01_01.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


/tmp/ipykernel_17957/3294081821.py:2: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True)


Not setting metadata
1800 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1800 events and 256 original time points ...
0 bad epochs dropped
Extracting EDF parameters from chb01_02.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


/tmp/ipykernel_17957/3294081821.py:2: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True)


Not setting metadata
1800 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1800 events and 256 original time points ...
0 bad epochs dropped
Extracting EDF parameters from chb01_03.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


/tmp/ipykernel_17957/3294081821.py:2: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True)


... (15 outputs left out)

/tmp/ipykernel_17957/3294081821.py:2: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True)


Not setting metadata
1800 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1800 events and 256 original time points ...
0 bad epochs dropped
Extracting EDF parameters from chb01_12.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


/tmp/ipykernel_17957/3294081821.py:2: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True)


Not setting metadata
1800 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1800 events and 256 original time points ...
0 bad epochs dropped
Train labels: (array([0, 1]), array([14250,   150]))
Test labels: (array([0, 1]), array([7050,  150]))


In [ ]:
# csp = CSP(n_components=4, reg='ledoit_wolf')

# X_train_csp = csp.fit_transform(X_train_raw, y_train)
# X_test_csp = csp.transform(X_test_raw)

In [ ]:
mean = X_train_raw.mean()
std = X_train_raw.std()

X_train_raw = (X_train_raw - mean) / std
X_test_raw = (X_test_raw - mean) / std

In [ ]:
# def create_sequences(X, y, seq_len=10):
#     X_seq = []
#     y_seq = []

#     for i in range(len(X) - seq_len):
#         seq = X[i:i+seq_len]
#         label_window = y[i:i+seq_len]


#         if np.any(label_window == 1):
#             label = 1
#         else:
#             label = 0

#         X_seq.append(seq)
#         y_seq.append(label)

#     return np.array(X_seq), np.array(y_seq)

In [ ]:
preictal_indices = np.where(y_train == 1)[0]
normal_indices = np.where(y_train == 0)[0]

In [ ]:
def data_generator(X, y, seq_len=10, batch_size=32):
    pre_idx = np.where(y == 1)[0]
    norm_idx = np.where(y == 0)[0]

    while True:
       X_batch = []
       y_batch = []

       for _ in range(batch_size):
           if np.random.rand() < 0.5:
               idx = np.random.choice(pre_idx)
           else:
               idx = np.random.choice(norm_idx)

           if idx + seq_len >= len(X):
               continue

           seq = X[idx:idx+seq_len]
           seq = seq.mean(axis=2)

           label_window = y[idx:idx+seq_len]
           label = 1 if np.any(label_window == 1) else 0

           X_batch.append(seq)
           y_batch.append(label)

       yield np.array(X_batch), np.array(y_batch)



In [ ]:
train_gen = data_generator(X_train_raw, y_train, seq_len=10, batch_size=32)
test_gen = data_generator(X_test_raw, y_test, seq_len=10, batch_size=32)

In [ ]:
# model = Sequential()

# model.add(LSTM(64, input_shape=(30, X_train_raw.shape[2])))
# model.add(Dense(1, activation='sigmoid'))

# model.compile(
#     optimizer='adam',
#     loss='binary_crossentropy',
#     metrics=['accuracy']
# )

# from sklearn.utils.class_weight import compute_class_weight

# weights = compute_class_weight(
#     class_weight='balanced',
#     classes=np.unique(y_train_seq),
#     y=y_train_seq
# )

# class_weights = {0: weights[0], 1: weights[1]}

# model.fit(
#     X_train_seq,
#     y_train_seq,
#     epochs=10,
#     batch_size=32,
#     validation_data=(X_test_seq, y_test_seq),
#     class_weight=class_weights
# )

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model = Sequential()

model.add(LSTM(64, input_shape=(10, X_train_raw.shape[1])))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_gen,
    steps_per_epoch=200,
    epochs=10
)

Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.7794 - loss: 0.4257
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9182 - loss: 0.2005
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.9520 - loss: 0.1358
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.9691 - loss: 0.1038
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.9833 - loss: 0.0622
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.9822 - loss: 0.0622
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.9898 - loss: 0.0400
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9906 - loss: 0.0370
Epoch 9/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.9945 - loss: 0.0243
Epoch 10/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9969 - loss: 0.0147


In [ ]:
all_preds = []
all_true = []

for _ in range(100):
    X_batch, y_batch = next(test_gen)

    preds = model.predict(X_batch)
    preds = (preds > 0.5).astype(int)

    all_preds.extend(preds.flatten())
    all_true.extend(y_batch)



print(confusion_matrix(all_true, all_preds))
print(classification_report(all_true, all_preds))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━